<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Decision.** Which existing page do I open first in a weekly content-refresh review?
**Unit of analysis.** One row = one page. **Output.** A ranked refresh-opportunity queue: score, suggested action, reason codes, confidence.
**Cost of a wrong call.** Refresh a page that is fine → wasted editor time and risk to a working page; miss a real decline → visibility erodes silently.
**Why data helps.** The signals interact and are heavy-tailed; the first honest fixed rule surfaced only 17 actionable pages out of 30,000.

In [1]:
import os, sys
import pandas as pd, numpy as np

if "google.colab" in sys.modules:
    os.chdir("flyrank-ml-internship-starter" if os.path.isdir("flyrank-ml-internship-starter") else ".")
else:
    for _ in range(3):
        if os.path.isdir("data/raw"):
            break
        os.chdir("..")
print("cwd:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("decision this work supports: which existing page to open first in a weekly refresh review")
print("unit of analysis: one row = one page; output = ranked refresh queue with actions + reason codes")
print("rows / columns / clients:", df.shape[0], "/", df.shape[1], "/", df["client_id"].nunique())
print("declining base rate:", round(df["is_declining_label"].mean(), 3))
print("trend states:", df["trend_direction"].value_counts().to_dict())

cwd: C:\Users\taher\OneDrive\Documents\first_assignment_FLYRANK\Fly2
decision this work supports: which existing page to open first in a weekly refresh review
unit of analysis: one row = one page; output = ranked refresh queue with actions + reason codes
rows / columns / clients: 30000 / 45 / 32
declining base rate: 0.542
trend states: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** Single anonymized starter table — `data/raw/content_refresh_anonymized.csv`, 30,000 rows × 44 columns, 32 pseudonymous clients. Public-safe: no client names, domains, URLs, titles, or keywords.
**Time geometry.** Content age 90–564 days; last update 1–373 days ago; 90-day impression window. No calendar survived anonymization → no seasonality; a single cross-sectional snapshot.
**Excluded on purpose.** `trend_direction` (the label), `trend_pct` (label source — alone it reaches AUC 1.000), `content_id` / `client_id` (identifiers — `client_id` only groups cross-validation), `provider_used` / `model_used` (not model features). `avg_position == 0` is re-coded as missing (1,205 rows).
**Systematic missingness.** Keyword fields are 100% missing for `feedly article` rows; `word_count` missing on 7,699 rows. Thinner-evidence rows get capped confidence, they are not silently dropped.

In [2]:
print("release: starter single-table release shipped with the repo (data/raw/content_refresh_anonymized.csv)")
print("public-safe: no client names, domains, URLs, titles, or keywords in the file")
print()
print("time geometry: content age", df["content_age_days"].min(), "-", df["content_age_days"].max(),
      "d | last update", df["days_since_last_update"].min(), "-", df["days_since_last_update"].max(),
      "d | 90-day impression window")
print()

excluded = {
    "trend_direction": "label (never a feature)",
    "trend_pct": "label source - lone single-tree AUC = 1.000 (pure leakage)",
    "content_id": "identifier",
    "client_id": "identifier - used ONLY to group cross-validation",
    "provider_used": "not a model feature per the data dictionary",
    "model_used": "not a model feature per the data dictionary",
}
for k, v in excluded.items():
    print(f"{k:16s} -> {v}")

print()
print("systematic missingness:")
print("  avg_position == 0 (no position data):", int((df["avg_position"] == 0).sum()))
print("  rows missing keyword fields:", int(df["search_volume"].isna().sum()))
print("  rows missing word_count / char_count:", int(df["word_count"].isna().sum()))
print("  feedly articles: keyword fields missing in",
      round(df[df["content_type"] == "feedly article"]["search_volume"].isna().mean(), 2), "of rows")
print()
sub = df[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]].dropna()
overlap = (sub["impressions_last_30d"] + sub["impressions_prev_30d"]) / sub["impressions_90d"].replace(0, np.nan)
print("90-day window overlap with the label's 30-day windows (measured):", round(overlap.mean(), 3))

release: starter single-table release shipped with the repo (data/raw/content_refresh_anonymized.csv)
public-safe: no client names, domains, URLs, titles, or keywords in the file

time geometry: content age 90 - 564 d | last update 1 - 373 d | 90-day impression window

trend_direction  -> label (never a feature)
trend_pct        -> label source - lone single-tree AUC = 1.000 (pure leakage)
content_id       -> identifier
client_id        -> identifier - used ONLY to group cross-validation
provider_used    -> not a model feature per the data dictionary
model_used       -> not a model feature per the data dictionary

systematic missingness:
  avg_position == 0 (no position data): 1205
  rows missing keyword fields: 2468
  rows missing word_count / char_count: 7699
  feedly articles: keyword fields missing in 1.0 of rows

90-day window overlap with the label's 30-day windows (measured): 0.562


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions.** Trend state is a reasonable proxy for "look here"; a 90-day snapshot can rank *within* a portfolio but not predict forward; features and label never share a window.
**Label.** `is_declining` = `trend_direction == "down"` (base rate 54.2%).
**Features.** 13 numeric + 8 categorical → 47-column vector. The crosstalk control is `impressions_earliest_slice` — the 90-day window minus the two 30-day windows the label is built from (measured overlap 0.562 removed), log-transformed.
**Baseline.** Hand rule: *"if a page is stale and still visible, check its CTR and engagement to pick a refresh action"* → `stale(≥180 d) × visible(≥500 impressions) × impressions_90d`. The bar the model must beat on the same folds.
**Model.** Random forest — 300 trees, max depth 8, balanced class weights, seed 42.
**Validation.** GroupKFold(5) on `client_id`: whole clients held out. A naive row split is wrong (pages of one client leak across folds); no time-aware split exists because no calendar survived.
**Leakage checks.** Single-tree per-feature AUC: `trend_pct` = 1.000 (excluded); every kept feature ≤ 0.62; `impressions_prev_30d` 0.623 flagged as a watch-item.

In [3]:
df["impressions_earliest_slice"] = (df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]).clip(lower=0)
df["log_impressions_earliest_slice"] = np.log1p(df["impressions_earliest_slice"])
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = ["search_volume", "competition", "cpc", "word_count", "char_count",
                    "log_impressions_earliest_slice", "content_age_days", "days_since_last_update",
                    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
categorical_features = ["competition_level", "content_type", "main_intent", "age_tier",
                        "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
X_num = df[numeric_features].fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown"))
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"].values
print("feature vector:", X.shape[1], "columns | label: is_declining = trend_direction == 'down'")

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
print()
print("per-feature leakage hunt (single-split tree AUC vs label):")
for col in ["trend_pct"] + numeric_features + ["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]:
    x = df[[col]].fillna(0)
    t = DecisionTreeClassifier(max_depth=1, random_state=42).fit(x, y)
    auc = roc_auc_score(y, t.predict_proba(x)[:, 1])
    flag = "LEAK" if auc > 0.75 else ("watch" if auc > 0.65 else "ok")
    print(f"  {col:30s} {auc:.3f} {flag}")

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["impressions_90d"]

df["low_ctr"] = (df["ctr"] < 1.0).astype(int)
df["low_eng"] = (df["engagement_rate"] < 20).astype(int)
def pick_action(row):
    if row["stale"] and row["visible"] and row["low_ctr"]:
        return "refresh_and_review_ctr"
    if row["stale"] and row["visible"] and row["low_eng"]:
        return "refresh_and_review_engagement"
    if row["stale"] and row["visible"]:
        return "refresh"
    return "monitor"
df["baseline_action"] = df.apply(pick_action, axis=1)
print()
print("baseline rule: stale(>=180d) x visible(>=500 impr) -> inspect CTR/engagement")
print("week-4 baseline actionable (non-monitor) pages:", int((df["baseline_action"] != "monitor").sum()))

feature vector: 47 columns | label: is_declining = trend_direction == 'down'



per-feature leakage hunt (single-split tree AUC vs label):
  trend_pct                      1.000 LEAK
  search_volume                  0.527 ok
  competition                    0.506 ok
  cpc                            0.501 ok
  word_count                     0.566 ok
  char_count                     0.563 ok
  log_impressions_earliest_slice 0.593 ok
  content_age_days               0.586 ok
  days_since_last_update         0.547 ok
  ctr                            0.540 ok
  avg_position                   0.545 ok
  engagement_rate                0.505 ok
  scroll_rate                    0.513 ok


  ai_traffic_pct                 0.502 ok
  impressions_90d                0.581 ok
  impressions_last_30d           0.532 ok
  impressions_prev_30d           0.623 ok



baseline rule: stale(>=180d) x visible(>=500 impr) -> inspect CTR/engagement
week-4 baseline actionable (non-monitor) pages: 17


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Measured before/after.** A naive row-level KFold reports P@50 ≈ 0.94 (AUC 0.751) — optimism from pages of the same client leaking across folds. Whole-client holdout lands at AUC 0.650 ± 0.054, P@50 0.732 ± 0.180.
**Model vs rule on the same folds.** P@20: 0.76 vs 0.66. P@50: 0.73 vs 0.61. Base rate 0.542. The model adds real but modest lift, and the top-50 of its queue holds a 0.76 declining rate — the honest number a reviewer can plan around.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import roc_auc_score


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def run_rf(train_idx, test_idx):
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y[train_idx])
    return rf.predict_proba(X.iloc[test_idx])[:, 1]


# BEFORE: naive row-level KFold (the optimistic design we averted)
naive_aucs, naive_p50 = [], []
for tr, te in KFold(n_splits=5, shuffle=True, random_state=42).split(X):
    proba = run_rf(tr, te)
    naive_aucs.append(roc_auc_score(y[te], proba))
    naive_p50.append(precision_at_k(proba, y[te], 50))

# AFTER: honest GroupKFold by client — also stores OOF probabilities for section 6
aucs, p20s, p50s, b20s, b50s = [], [], [], [], []
oof = np.zeros(len(df))
for tr, te in GroupKFold(n_splits=5).split(X, y, groups=df["client_id"]):
    proba = run_rf(tr, te)
    oof[te] = proba
    aucs.append(roc_auc_score(y[te], proba))
    p20s.append(precision_at_k(proba, y[te], 20))
    p50s.append(precision_at_k(proba, y[te], 50))
    b = df["baseline_score"].values[te]
    b20s.append(precision_at_k(b, y[te], 20))
    b50s.append(precision_at_k(b, y[te], 50))

df["oof_decline_prob"] = oof

print("BEFORE (naive row KFold):   AUC %.3f +- %.3f | P@50 %.3f +- %.3f" % (
    np.mean(naive_aucs), np.std(naive_aucs), np.mean(naive_p50), np.std(naive_p50)))
print("AFTER  (grouped by client): AUC %.3f +- %.3f | P@20 %.3f | P@50 %.3f +- %.3f" % (
    np.mean(aucs), np.std(aucs), np.mean(p20s), np.mean(p50s), np.std(p50s)))
print("RULE   (same grouped folds): P@20 %.3f | P@50 %.3f | base rate %.3f" % (
    np.mean(b20s), np.mean(b50s), y.mean()))
print("model queue top-50 declining rate (OOF): %.3f" % precision_at_k(oof, y, 50))
print()

pd.DataFrame({
    "metric": ["roc_auc", "precision_at_20", "precision_at_50"],
    "naive_row_kfold": [round(float(np.mean(naive_aucs)), 3), None, round(float(np.mean(naive_p50)), 3)],
    "grouped_by_client": [round(float(np.mean(aucs)), 3), round(float(np.mean(p20s)), 3),
                          round(float(np.mean(p50s)), 3)],
    "grouped_baseline_rule": [None, round(float(np.mean(b20s)), 3), round(float(np.mean(b50s)), 3)],
})

BEFORE (naive row KFold):   AUC 0.751 +- 0.005 | P@50 0.940 +- 0.013
AFTER  (grouped by client): AUC 0.650 +- 0.054 | P@20 0.760 | P@50 0.732 +- 0.180
RULE   (same grouped folds): P@20 0.660 | P@50 0.612 | base rate 0.542
model queue top-50 declining rate (OOF): 0.760



,metric,naive_row_kfold,grouped_by_client,grouped_baseline_rule
0,roc_auc,0.751,0.650,NaN
1,precision_at_20,NaN,0.760,0.660
2,precision_at_50,0.940,0.732,0.612


## 5. Limitations

*What this work cannot claim.*

This work cannot claim: a **causal** refresh impact (no intervention, no calendar — a refresh stays a human decision, not a predicted guarantee); a **forward** outlook (one 90-day snapshot, cross-sectional); that rows with missing position / keyword / content evidence carry full reason codes; that 32 uneven clients generalize per-client (one client supplies 96 of the top-100 queue rows — measured in §6 — and grouped P@50 varies ± 0.18); or anything about **Google's algorithm** (this ranks pages inside one pseudonymized portfolio only). Language stays observed / measured / directional / decision-support.

In [5]:
print("rows without position data:", int(df["avg_position"].isna().sum()))
print("rows without keyword data:", int(df["search_volume"].isna().sum()))
print("rows missing word_count / char_count:", int(df["word_count"].isna().sum()))
print("no calendar -> seasonality untestable; single 90-day snapshot -> cross-sectional only")
print()
print("per-client performance varies: grouped P@50 std across folds = %.3f" % np.std(p50s))
print("grouped AUC across folds: min %.3f max %.3f" % (min(aucs), max(aucs)))
print("client concentration at the top of the queue is measured in section 6")

rows without position data: 1205
rows without keyword data: 2468
rows missing word_count / char_count: 7699
no calendar -> seasonality untestable; single 90-day snapshot -> cross-sectional only

per-client performance varies: grouped P@50 std across folds = 0.180
grouped AUC across folds: min 0.576 max 0.743
client concentration at the top of the queue is measured in section 6


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

A ranked queue → action + reason codes + confidence per page. A reviewer works top-down; `monitor` = "watch, don't touch", not "safe".
Priority: `refresh_and_review_ctr` > `refresh_and_review_engagement` > `refresh` > `expand_and_refresh` > `monitor`.
A human checks before acting: is the trend real, is the reason code the real problem (e.g. a SERP feature stealing clicks), is the tracking intact, was the last update logged, did the model have the evidence.
Never automated: no autopublish / unpublish, no deletes from a score, no treating `declining_with_demand` as caused by us, no scoring brand-new pages (<90 d) or no-data rows, no "predicted Google".
Keeping it fresh: re-score ~4 weeks; retrain when honest P@50 < ~0.60 or top-50 declining rate < base + 0.10; alert when one client holds > 1/3 of the top-100.

In [6]:
# Transparent reason codes -> action -> confidence, sitting on the grouped OOF probability.
visible = df["impressions_90d"] >= 500
df["stale_visible"] = ((df["days_since_last_update"] >= 180) & visible).astype(int)
df["declining_with_demand"] = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).astype(int)
df["thin_visible"] = ((df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)).astype(int)
df["page_one_decay"] = ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).astype(int)
df["low_ctr_visible"] = (visible & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).astype(int)
df["low_eng_visible"] = ((df["sessions_90d"] >= 30) &
                         (((df["engagement_rate"] > 0) & (df["engagement_rate"] < 30)) |
                          ((df["scroll_rate"] > 0) & (df["scroll_rate"] < 30)))).astype(int)

log_imp = np.log1p(df["impressions_90d"])
visibility = (log_imp - log_imp.min()) / (log_imp.max() - log_imp.min())
df["baseline_refresh_score"] = (0.40 * visibility
    + 0.30 * df["declining_with_demand"] * visibility
    + 0.20 * np.clip(df["days_since_last_update"] / 180, 0, 1)
    + 0.10 * ((df["low_ctr_visible"] + df["low_eng_visible"]) / 2)).clip(0, 1)
df["final_refresh_score"] = 100 * (0.70 * df["oof_decline_prob"] + 0.30 * df["baseline_refresh_score"]).clip(0, 1)


def reason_codes(row):
    reasons = []
    if row["stale_visible"]:
        reasons.append("stale_visible_page")
    if row["declining_with_demand"]:
        reasons.append("declining_with_demand")
    if row["thin_visible"]:
        reasons.append("thin_visible_page")
    if row["page_one_decay"]:
        reasons.append("page_one_decay_risk")
    if row["low_ctr_visible"]:
        reasons.append("low_ctr_visible_page")
    if row["low_eng_visible"]:
        reasons.append("low_engagement_visible_page")
    if row["oof_decline_prob"] >= 0.65:
        reasons.append("model_decline_risk")
    if row["oof_decline_prob"] >= 0.5 and row["impressions_90d"] >= 500:
        reasons.append("visible_model_opportunity")
        if row["low_ctr_visible"]:
            reasons.append("ctr_review_candidate")
        if row["low_eng_visible"]:
            reasons.append("engagement_review_candidate")
    return "|".join(reasons) if reasons else "general_refresh_review"


df["final_reason_codes"] = df.apply(reason_codes, axis=1)


def suggested_action(row):
    reasons = set(row["final_reason_codes"].split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "ctr_review_candidate" in reasons and ("model_decline_risk" in reasons or "declining_with_demand" in reasons):
        return "refresh_and_review_ctr"
    if "engagement_review_candidate" in reasons and ("model_decline_risk" in reasons or "declining_with_demand" in reasons):
        return "refresh_and_review_engagement"
    if {"model_decline_risk", "declining_with_demand", "stale_visible_page", "visible_model_opportunity"} & reasons:
        return "refresh"
    return "monitor"


df["suggested_action"] = df.apply(suggested_action, axis=1)

high_t = df["final_refresh_score"].quantile(0.8)
med_t = df["final_refresh_score"].quantile(0.5)


def confidence(row):
    if (row["final_refresh_score"] >= high_t and row["impressions_90d"] >= 500
            and row["sessions_90d"] >= 10 and row["oof_decline_prob"] >= 0.5):
        return "high"
    if row["final_refresh_score"] >= med_t:
        return "medium"
    return "low"


df["confidence"] = df.apply(confidence, axis=1)

queue = df.sort_values(["final_refresh_score", "impressions_90d", "sessions_90d"],
                       ascending=[False, False, False]).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

print("action counts:", df["suggested_action"].value_counts().to_dict())
print("confidence counts:", df["confidence"].value_counts().to_dict())

misfire = (df["oof_decline_prob"] >= 0.65) & (df["trend_direction"] != "down")
print("model-vs-label misfire candidates for a human eye:", int(misfire.sum()),
      "| of which on visible pages:", int((misfire & (df["impressions_90d"] >= 500)).sum()))

conc = queue.head(100)["client_id"].value_counts()
print("top-100 client concentration: max share", int(conc.max()), "/ 100 across", int(conc.nunique()), "clients")
print()

queue.head(5)[["final_rank", "content_id", "final_refresh_score", "confidence",
               "suggested_action", "final_reason_codes", "trend_direction"]]

action counts: {'monitor': 11540, 'refresh': 11010, 'refresh_and_review_ctr': 5633, 'refresh_and_review_engagement': 1735, 'expand_and_refresh': 82}
confidence counts: {'low': 15000, 'medium': 11077, 'high': 3923}
model-vs-label misfire candidates for a human eye: 1461 | of which on visible pages: 1025
top-100 client concentration: max share 96 / 100 across 2 clients



,final_rank,content_id,final_refresh_score,confidence,suggested_action,final_reason_codes,trend_direction
0,1,content_f988b4cba4ea,77.332209,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
1,2,content_b60c70399545,76.414589,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
2,3,content_dc07a16ea110,76.217884,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
3,4,content_813e88069237,76.152767,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
4,5,content_4f5826036689,76.117180,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper embeds four charts plus the honest results table. This cell regenerates the exact same charts to `work/outputs/capstone_*.svg`, so the notebook and the page stay one source of truth.

In [7]:
os.makedirs("work/outputs", exist_ok=True)


def svg_bars(title, labels0, values, path, color="#426B69", digits=0):
    labels = [str(x) for x in labels0]
    values = [float(v) for v in values]
    m = max(max(values, default=1), 1)
    W, H, ml, mt, mb, gap = 860, 300, 250, 46, 18, 8
    pw, ph = W - ml - 40, H - mt - mb
    bh = max(16.0, (ph - gap * max(len(values) - 1, 0)) / max(len(values), 1))
    fmt = "{:,.3f}" if digits else "{:,.0f}"
    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{W / 2}" y="30" text-anchor="middle" font-family="Arial" font-size="16" fill="#16232a">{title}</text>',
    ]
    for i, (label, value) in enumerate(zip(labels, values)):
        y = mt + i * (bh + gap)
        bw = max((value / m) * pw, 2.0)
        lines.append(f'<text x="{ml - 12}" y="{y + bh * 0.68:.1f}" text-anchor="end" font-family="Arial" font-size="12.5" fill="#27343b">{label}</text>')
        lines.append(f'<rect x="{ml}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{ml + bw + 8:.1f}" y="{y + bh * 0.68:.1f}" font-family="Arial" font-size="12.5" fill="#16232a">{fmt.format(value)}</text>')
    lines.append("</svg>")
    with open(path, "w") as f:
        f.write("\n".join(lines))


svg_bars("Validation design - Precision@50",
         ["Naive row KFold", "Grouped by client", "Base rate"],
         [0.94, 0.732, 0.542], "work/outputs/capstone_validation_p50.svg", color="#B8C0C4", digits=3)
svg_bars("Model vs rule (same grouped folds)",
         ["Rule P@20", "Model P@20", "Rule P@50", "Model P@50"],
         [0.66, 0.76, 0.612, 0.732], "work/outputs/capstone_model_vs_rule.svg", color="#6F4E7C", digits=3)
svg_bars("Action mix", df["suggested_action"].value_counts().index.tolist(),
         df["suggested_action"].value_counts().values.tolist(),
         "work/outputs/capstone_action_mix.svg", color="#426B69")
svg_bars("Confidence mix", ["low", "medium", "high"],
         [int((df["confidence"] == c).sum()) for c in ["low", "medium", "high"]],
         "work/outputs/capstone_confidence_mix.svg", color="#6F4E7C")

print("artifacts written to work/outputs/:")
for name in sorted(n for n in os.listdir("work/outputs") if n.startswith("capstone_")):
    print(" -", name)

artifacts written to work/outputs/:
 - capstone_action_mix.svg
 - capstone_confidence_mix.svg
 - capstone_model_vs_rule.svg
 - capstone_validation_p50.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.